In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM


df = pd.read_csv(
    r"C:\A-CMSI research\Data\HMM data\HMM_Input_features.csv"
)

# =====================================
# Features
# =====================================

features = [
    "SPY_Return",
    "SPY_V",
    "RollingVol21",
    "RollingSkew21",
    "Drawdown",
    "VolOfVol",
    "VIX",
    "C_t"
]

X = df[features]


scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


best_model = None
best_score = -np.inf

for seed in range(30):

    model = GaussianHMM(
        n_components=4,
        covariance_type="full",
        n_iter=200,
        random_state=seed
    )

    model.fit(X_scaled)

    score = model.score(X_scaled)

    if score > best_score:

        best_score = score
        best_model = model

print("="*60)
print("Best Log-Likelihood:", best_score)
print("="*60)


hidden_states = best_model.predict(X_scaled)

df["State"] = hidden_states


print("\nSTATE MEANS\n")

print(
    df.groupby("State")[[
        "SPY_Return",
        "SPY_V",
        "RollingVol21",
        "RollingSkew21",
        "Drawdown",
        "VolOfVol",
        "VIX",
        "C_t"
    ]].mean()
)


print("\nTRANSITION MATRIX\n")

print(best_model.transmat_)


logL = best_model.score(X_scaled)

n_states = best_model.n_components
n_features = X_scaled.shape[1]
n_samples = X_scaled.shape[0]


k = (
    (n_states - 1)
    + n_states * (n_states - 1)
    + n_states * n_features
    + n_states * n_features * (n_features + 1) / 2
)

AIC = -2 * logL + 2 * k

BIC = -2 * logL + np.log(n_samples) * k

print("\nMODEL SELECTION")

print("----------------------------")
print("Number of States :", n_states)
print("Log-Likelihood   :", logL)
print("Parameters       :", int(k))
print("AIC              :", AIC)
print("BIC              :", BIC)
print("----------------------------")

Best Log-Likelihood: -13754.413846617503

STATE MEANS

       SPY_Return     SPY_V  RollingVol21  RollingSkew21  Drawdown  VolOfVol  \
State                                                                          
0        0.001192 -0.710628      0.005412      -0.013821 -0.003413  0.000627   
1        0.000300  0.509138      0.011650      -0.135103 -0.087028  0.001433   
2       -0.000881  0.615897      0.019216      -0.474699 -0.086750  0.004669   
3        0.000808 -0.518769      0.006702      -0.240411 -0.010968  0.001199   

             VIX       C_t  
State                       
0      12.592766  0.530126  
1      20.234276  0.759792  
2      30.883333  0.794640  
3      17.110282  0.575511  

TRANSITION MATRIX

[[9.70335297e-01 3.67920163e-03 3.23865004e-03 2.27468516e-02]
 [3.71008095e-03 9.79995312e-01 4.52017907e-03 1.17744277e-02]
 [3.47985664e-93 3.52072873e-02 9.43446357e-01 2.13463562e-02]
 [2.30743122e-02 7.85523786e-03 1.18834024e-02 9.57187048e-01]]

MODEL SELECTION
